In [2]:
from pathlib import Path
import os, sys, subprocess, importlib.metadata
IS_COLAB = Path('/content').is_dir() and 'google.colab' in sys.modules
if IS_COLAB:
    ROOT = Path('/content/masar-modern-data-engineering')
    if not ROOT.exists():
        subprocess.run(['git','clone','https://github.com/almiyead-rgb/masar-modern-data-engineering.git',str(ROOT)], check=True)
    pins = {'pyspark':'3.5.8','delta-spark':'3.3.3','py4j':'0.10.9.9'}
    missing = []
    for package, version in pins.items():
        try: observed = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError: observed = None
        if observed != version: missing.append(f'{package}=={version}')
    if missing:
        subprocess.run([sys.executable,'-m','pip','install','--quiet',*missing],check=True)
    candidates = list(Path('/usr/lib/jvm').glob('*17*'))
    if not any((p/'bin/java').is_file() for p in candidates):
        subprocess.run(['apt-get','update','-qq'],check=True)
        subprocess.run(['apt-get','install','-y','-qq','openjdk-17-jre-headless'],check=True)
        candidates = list(Path('/usr/lib/jvm').glob('*17*'))
    os.environ['JAVA_HOME'] = str(next(p for p in candidates if (p/'bin/java').is_file()))
    os.chdir(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p/'course.json').is_file()),None)
    if ROOT is None: raise FileNotFoundError('Open from the repository root; see docs/SETUP.md')
print('Repository:', ROOT)
print('Python:', sys.version.split()[0])


Repository: /content/masar-modern-data-engineering
Python: 3.11.13


In [3]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src" / "masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
(ROOT / "outputs").mkdir(exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="day01_", dir=ROOT / "outputs"))
print("Inspect the source and assumptions before running the next native section.")

Inspect the source and assumptions before running the next native section.


In [4]:
from masar.sources import verify_manifest, load_sources, profile_sources
SOURCE = ROOT / "data" / "masar-small-v1"
manifest = verify_manifest(SOURCE)
feeds = load_sources(SOURCE)
print("Dataset:", manifest["label"])
print("Verified manifest files:", len(manifest["files"]))
print(json.dumps({name: len(rows) for name, rows in feeds.items()}, indent=2))
print("Trip columns:", list(feeds["trips"][0]))
print("First synthetic event:", json.dumps(feeds["gps_events"][0], ensure_ascii=False))

Dataset: MASAR_SMALL_V1
Verified manifest files: 10
{
  "trips": 72,
  "drivers": 6,
  "gps_events": 216
}
Trip columns: ['trip_id', 'driver_id', 'city', 'start_ts', 'end_ts', 'fare_sar', 'distance_km']
First synthetic event: {"city": "Riyadh", "event_id": "SYN_E0001_0", "event_ts": "2026-06-01T06:00:00+03:00", "location": {"lat": 24.7, "lon": 46.7}, "synthetic": true, "trip_id": "SYN_T0001"}


In [5]:
result = profile_sources(SOURCE)
print(json.dumps(result["profile"], indent=2))
print(json.dumps(result["relations"], indent=2))
print(json.dumps(result["city_profile"], indent=2))

{
  "trips": {
    "rows": 72,
    "key": "trip_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "city": 0,
      "distance_km": 0,
      "driver_id": 0,
      "end_ts": 0,
      "fare_sar": 0,
      "start_ts": 0,
      "trip_id": 0
    }
  },
  "drivers": {
    "rows": 6,
    "key": "driver_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "driver_id": 0,
      "driver_rating": 0,
      "vehicle_type": 0
    }
  },
  "gps_events": {
    "rows": 216,
    "key": "event_id",
    "duplicate_key_groups": 0,
    "duplicate_excess_rows": 0,
    "missing_top_level_fields": {
      "city": 0,
      "event_id": 0,
      "event_ts": 0,
      "location": 0,
      "synthetic": 0,
      "trip_id": 0
    }
  }
}
{
  "trips_without_driver": 0,
  "events_without_trip": 0,
  "events_with_invalid_coordinates": 0
}
{
  "original_labels": {
    " dammam ": 3,
    " jeddah ": 3,
    " riyad

In [6]:
if not all(result["checks"].values()):
    raise AssertionError(result["checks"])
output = RUN / "source_inspection.json"
output.write_text(json.dumps(result, ensure_ascii=False, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print(json.dumps(result["checks"], indent=2))
print("PASS: source inspection only")
print("Saved:", output.name)
print("Source inspection complete. Continue to the Bronze section.")

{
  "source_counts": true,
  "unique_base_keys": true,
  "base_top_level_complete": true,
  "valid_links_and_coordinates": true,
  "three_events_per_trip": true,
  "base_city_set": true
}
PASS: source inspection only
Saved: source_inspection.json
Source inspection complete. Continue to the Bronze section.


In [7]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.runtime import inspect_environment, require_environment, start_spark
print(json.dumps(inspect_environment(), indent=2))

{
  "scope": "DEPENDENCY_PREFLIGHT_ONLY",
  "python": "3.11.13",
  "java": "openjdk version \"17.0.20\" 2026-07-21",
  "java_major": 17,
  "packages": {
    "pyspark": {
      "required": "3.5.8",
      "observed": "3.5.8"
    },
    "delta-spark": {
      "required": "3.3.3",
      "observed": "3.3.3"
    },
    "py4j": {
      "required": "0.10.9.9",
      "observed": "0.10.9.9"
    }
  },
  "status": "DEPENDENCIES_PRESENT_ENGINE_NOT_TESTED",
  "issues": [],
  "engine_executed": false
}


In [8]:
require_environment()
from masar.workspace import new_workspace, require_fixed_dataset, write_json, workspace_path
require_fixed_dataset(SOURCE)
WORK = new_workspace(ROOT, "day01_bronze")
print("Workspace:", WORK.relative_to(ROOT))

Workspace: outputs/day01_bronze_u0g5pn_d


In [9]:
from masar.bronze import raw_frame, ingest_feed, verify_bronze
# Functions are imported without starting Spark. The next cell executes the lab.
print("Bronze functions loaded. The next cell writes and reads the Delta tables.")

Bronze functions loaded. The next cell writes and reads the Delta tables.


In [10]:
from masar.workspace import record_bronze_success
spark = start_spark(WORK)
try:
    print("Spark:", spark.version)
    raw_trips = raw_frame(spark, SOURCE, "trips")
    raw_trips.printSchema()
    raw_trips.show(3, truncate=False)
    print("Source trips:", raw_trips.count())
    for feed in ("trips", "drivers", "gps_events"):
        print(ingest_feed(spark, SOURCE, WORK, feed, "base_001"))
    print(ingest_feed(spark, SOURCE, WORK, "trips", "replay_002"))
    report = verify_bronze(spark, SOURCE, WORK)
    record_bronze_success(ROOT, WORK)
    print(json.dumps({"counts": report["counts"], "checks": report["checks"]}, indent=2))
    print("Retained:", (WORK / "reports/bronze.json").relative_to(ROOT))
finally:
    spark.stop()

Spark: 3.5.8
root
 |-- trip_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- city: string (nullable = true)
 |-- start_ts: string (nullable = true)
 |-- end_ts: string (nullable = true)
 |-- fare_sar: string (nullable = true)
 |-- distance_km: string (nullable = true)

+---------+---------+------+-------------------------+-------------------------+--------+-----------+
|trip_id  |driver_id|city  |start_ts                 |end_ts                   |fare_sar|distance_km|
+---------+---------+------+-------------------------+-------------------------+--------+-----------+
|SYN_T0001|SYN_D001 |Riyadh|2026-06-01T06:00:00+03:00|2026-06-01T06:08:00+03:00|18.00   |3.50       |
|SYN_T0002|SYN_D002 |Riyadh|2026-06-01T08:00:00+03:00|2026-06-01T08:11:00+03:00|19.25   |3.85       |
|SYN_T0003|SYN_D001 |Riyadh|2026-06-01T10:00:00+03:00|2026-06-01T10:14:00+03:00|20.50   |4.20       |
+---------+---------+------+-------------------------+-------------------------+--------+---

In [11]:
pwd

'/content/masar-modern-data-engineering'

In [12]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src" / "masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
(ROOT / "outputs").mkdir(exist_ok=True)
RUN = Path(tempfile.mkdtemp(prefix="day01_", dir=ROOT / "outputs"))
print("Inspect the source and assumptions before running the next native section.")

Inspect the source and assumptions before running the next native section.


In [13]:
from decimal import Decimal
from masar.cost import DEFAULTS, evaluate_cost, serializable, cost_report
print(json.dumps(DEFAULTS, indent=2))
result = cost_report()
print(json.dumps(result["base"], indent=2))

{
  "days": "30",
  "hours_per_day": "24",
  "cores": "4",
  "price_per_core_hour": "0.50",
  "storage_gib": "100",
  "storage_price_per_gib_month": "0.20",
  "work_hours_per_day": "2",
  "startup_hours_per_day": "0.25",
  "scheduled_extra_monthly": "30",
  "always_on_extra_monthly": "0"
}
{
  "unit": "TU (hypothetical teaching units; not currency)",
  "always_on_hours": "720",
  "scheduled_hours": "67.50",
  "always_on_compute": "1440.00",
  "scheduled_compute": "135.0000",
  "storage_each": "20.00",
  "always_on_total": "1460.00",
  "scheduled_total": "185.0000",
  "difference": "1275.0000",
  "reduction_fraction": "0.8732876712328767123287671233",
  "break_even_work_hours_per_day": "23.25"
}


In [14]:
print("Work h/day | Always-on TU | Scheduled TU | Difference TU")
for row in result["sensitivity"]:
    print(f"{row['work_hours_per_day']:>10} | {row['always_on_total']:>12} | {row['scheduled_total']:>12} | {row['difference']:>13}")

Work h/day | Always-on TU | Scheduled TU | Difference TU
         0 |      1460.00 |        50.00 |       1410.00
         1 |      1460.00 |     125.0000 |     1335.0000
         2 |      1460.00 |     185.0000 |     1275.0000
         4 |      1460.00 |     305.0000 |     1155.0000
         8 |      1460.00 |     545.0000 |      915.0000
        12 |      1460.00 |     785.0000 |      675.0000
        18 |      1460.00 |    1145.0000 |      315.0000
        23 |      1460.00 |    1445.0000 |       15.0000
     23.25 |      1460.00 |    1460.0000 |        0.0000
     23.75 |      1460.00 |    1490.0000 |      -30.0000


In [15]:
base = evaluate_cost(DEFAULTS)
checks = {
    "base_totals": (base["always_on_total"], base["scheduled_total"]) == (Decimal("1460"), Decimal("185")),
    "break_even": evaluate_cost({**DEFAULTS, "work_hours_per_day": "23.25"})["difference"] == 0,
    "counterexample": evaluate_cost({**DEFAULTS, "work_hours_per_day": "23.75"})["difference"] < 0,
    "zero_rate": evaluate_cost({**DEFAULTS, "price_per_core_hour": "0"})["break_even_work_hours_per_day"] is None,
}
try:
    evaluate_cost({**DEFAULTS, "cores": "-1"})
except ValueError:
    checks["negative_input_rejected"] = True
else:
    checks["negative_input_rejected"] = False
if not all(checks.values()):
    raise AssertionError(checks)
result["checks"] = checks
print(json.dumps(checks, indent=2))

{
  "base_totals": true,
  "break_even": true,
  "counterexample": true,
  "zero_rate": true,
  "negative_input_rejected": true
}


In [16]:
output = RUN / "cost_model_result.json"
output.write_text(json.dumps(result, sort_keys=True, indent=2) + "\n", encoding="utf-8")
print("PASS: hypothetical cost arithmetic only")
print("Saved:", output.name)
print("Cost arithmetic complete. Continue to the measured Spark comparison.")

PASS: hypothetical cost arithmetic only
Saved: cost_model_result.json
Cost arithmetic complete. Continue to the measured Spark comparison.


In [17]:
from pathlib import Path
import sys, json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from within the complete course repository")
sys.path.insert(0, str(ROOT / "src"))
SOURCE = ROOT / "data/masar-small-v1"
from masar.runtime import inspect_environment, require_environment, start_spark
print(json.dumps(inspect_environment(), indent=2))

{
  "scope": "DEPENDENCY_PREFLIGHT_ONLY",
  "python": "3.11.13",
  "java": "openjdk version \"17.0.20\" 2026-07-21",
  "java_major": 17,
  "packages": {
    "pyspark": {
      "required": "3.5.8",
      "observed": "3.5.8"
    },
    "delta-spark": {
      "required": "3.3.3",
      "observed": "3.3.3"
    },
    "py4j": {
      "required": "0.10.9.9",
      "observed": "0.10.9.9"
    }
  },
  "status": "DEPENDENCIES_PRESENT_ENGINE_NOT_TESTED",
  "issues": [],
  "engine_executed": false
}


In [18]:
require_environment()
from masar.workspace import completed_bronze_workspace, require_fixed_dataset
require_fixed_dataset(SOURCE)
WORK = completed_bronze_workspace(ROOT)
spark = start_spark(WORK)
print("Workspace:", WORK.relative_to(ROOT))

Workspace: outputs/day01_bronze_u0g5pn_d


In [19]:
from masar.benchmark import benchmark
try:
    report = benchmark(spark, SOURCE, WORK, repetitions=4)
    print(json.dumps(report["expected_and_observed_aggregate"], indent=2))
    print(json.dumps(report["measurements"], indent=2))
    print("Plans:", report["plans"])
    print("Evidence:", (WORK / "reports/benchmark.json").relative_to(ROOT))
finally:
    spark.stop()

{
  "rows": 72,
  "nonnull_fares": 72,
  "fare_total": "1794.60"
}
{
  "csv": {
    "samples_s": [
      0.20014098799993008,
      0.3671114670000861,
      0.33572564699989016,
      0.29464423599984
    ],
    "median_s": 0.3151849414998651,
    "min_s": 0.20014098799993008,
    "max_s": 0.3671114670000861
  },
  "delta_v0": {
    "samples_s": [
      1.6964692350002224,
      2.052133282000341,
      2.1450002330002462,
      1.6483117460002177
    ],
    "median_s": 1.8743012585002816,
    "min_s": 1.6483117460002177,
    "max_s": 2.1450002330002462
  }
}
Plans: {'csv': 'reports/plans/csv.txt', 'delta_v0': 'reports/plans/delta_v0.txt'}
Evidence: outputs/day01_bronze_u0g5pn_d/reports/benchmark.json


In [20]:
# DAY01_HANDOFF_V2: retain the pointer and all small reports as well as Delta files.
from pathlib import Path
import zipfile
from masar.workspace import completed_bronze_workspace
WORK = completed_bronze_workspace(ROOT)
pointer = ROOT / 'outputs/day01_bronze_success.json'
files_to_save = {pointer, *(p for p in WORK.rglob('*') if p.is_file())}
for name in ('source_inspection.json', 'cost_model_result.json'):
    files_to_save.update((ROOT / 'outputs').rglob(name))
archive = ROOT / 'outputs/day01_handoff.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(files_to_save):
        bundle.write(path, arcname=path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
    assert bundle.read('outputs/day01_bronze_success.json') == pointer.read_bytes()
    assert any(name.endswith('source_inspection.json') for name in bundle.namelist())
    assert any(name.endswith('cost_model_result.json') for name in bundle.namelist())
print('Keep this ZIP for the next day:', archive)
print('Also save this notebook with outputs and your LAB01/LAB02 notes.')
if IS_COLAB:
    from google.colab import files
    files.download(str(archive))

Keep this ZIP for the next day: /content/masar-modern-data-engineering/outputs/day01_handoff.zip
Also save this notebook with outputs and your LAB01/LAB02 notes.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>